In [14]:
import pandas as pd
from langdetect import detect
import string
from vllm import LLM, SamplingParams
from openai import OpenAI
from tqdm import tqdm


In [2]:
df = pd.read_csv('nivi_user_questions.csv')
user_questions_list = df['Answers/Response'].tolist()


In [5]:
len(user_questions_list)

5619

In [4]:
def is_english_only(text):
    """
    Check if text contains only English characters, spaces, and basic punctuation.
    """
    allowed_chars = set(string.ascii_letters + string.digits + string.punctuation + string.whitespace)
    return all(char in allowed_chars for char in text)

def is_english_language(text):
    """
    Use langdetect library to determine if text is in English.
    """
    try:
        detected_language = detect(text)
        return detected_language == 'en'
    except:
        return False

def analyze_question_clarity(llm, question, sampling_params):
    """
    Use LLM to determine if a question is well-specified or too vague.
    If vague, generate a clarifying question.
    """

    clarity_prompt = f"""You are a medical health chatbot answering user's questions about pregnancy. Analyze the following question and determine if it's well-specified or too vague to answer properly.

    Question: "{question}"

    Instructions:
    - If the question is well-specified (clear, specific, and answerable), respond with: "WELL_SPECIFIED"
    - If the question is too vague (lacks context, specificity, or is ambiguous), respond with: "VAGUE"

    Response:"""

    clarifying_prompt = f"""You are a medical health chatbot answering user's questions about pregnancy. The following user question about pregnancy is too vague to answer accurately. Generate a clarifying question that would help make it more specific and answerable.

    Original question: "{question}"

    Generate ONE clarifying question that would help gather the missing information needed to answer this question properly. Make the clarifying question clear and specific.

    Clarifying question:"""

    clarity_response = llm.generate([clarity_prompt], sampling_params, use_tqdm=False)[0]
    clarity_result = clarity_response.outputs[0].text.strip()
    
    if "VAGUE" in clarity_result.upper():
        clarifying_response = llm.generate([clarifying_prompt], sampling_params, use_tqdm=False)[0]
        clarifying_question = clarifying_response.outputs[0].text.strip()
        return {
            "original_question": question,
            "is_well_specified": False,
            "clarifying_question": clarifying_question
        }
    else:
        return {
            "original_question": question,
            "is_well_specified": True,
            "clarifying_question": None
        }

def process_questions_with_llm(filtered_questions_list, llm, sampling_params, num_questions=30):
    # Sort questions by length, longest to shortest
    sorted_questions = sorted(filtered_questions_list, key=len, reverse=True)
    questions_to_process = sorted_questions[:num_questions]
    results = []
    
    for i, question in enumerate(questions_to_process, 1):
        try:
            result = analyze_question_clarity(llm, question, sampling_params)
            results.append(result)
            
        except Exception as e:
            print(f"Error processing question: {e}")
            results.append({
                "original_question": question,
                "is_well_specified": None,
                "clarifying_question": f"Error: {str(e)}"
            })
    
    return results

def filter_english_questions(user_questions_list):
    """
    Filter questions to include only English-only questions longer than 3 words.
    """
    filtered_questions = []
    
    for question in user_questions_list:
        # Skip empty or very short strings
        if not question or not question.strip():
            continue
            
        # Check word count (longer than 10 words)
        words = question.strip().split()
        if len(words) <= 10:
            continue
            
        # Check if contains only English characters
        if not is_english_only(question):
            continue
            
        # Check if it's detected as English language
        if not is_english_language(question):
            continue
            
        filtered_questions.append(question)
    
    return filtered_questions


In [6]:
filtered = filter_english_questions(user_questions_list)

In [5]:

llm = LLM(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        tensor_parallel_size=1, 
        dtype="float16",  
        max_model_len=4096  
)

sampling_params = SamplingParams(
        temperature=0.7, 
        max_tokens=100,
        stop=["\n", "Question:", "Response:"]
    )

WARNING 07-06 14:02:39 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 07-06 14:02:51 [config.py:717] This model supports multiple tasks: {'reward', 'score', 'classify', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 07-06 14:02:51 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-06 14:02:53 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='meta-llama/Meta-Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 07-06 14:03:18 [loader.py:458] Loading weights took 22.57 seconds
INFO 07-06 14:03:18 [gpu_model_runner.py:1347] Model loading took 14.9889 GiB and 23.457606 seconds
INFO 07-06 14:03:27 [backends.py:420] Using cache directory: /home/gliu2/.cache/vllm/torch_compile_cache/3529aba791/rank_0_0 for vLLM's torch.compile
INFO 07-06 14:03:27 [backends.py:430] Dynamo bytecode transform time: 9.53 s
INFO 07-06 14:03:36 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 7.556 s
INFO 07-06 14:03:37 [monitor.py:33] torch.compile takes 9.53 s in total
INFO 07-06 14:03:39 [kv_cache_utils.py:634] GPU KV cache size: 213,872 tokens
INFO 07-06 14:03:39 [kv_cache_utils.py:637] Maximum concurrency for 4,096 tokens per request: 52.21x
INFO 07-06 14:04:09 [gpu_model_runner.py:1686] Graph capturing finished in 31 secs, took 0.51 GiB
INFO 07-06 14:04:09 [core.py:159] init engine (profile, create kv cache, warmup model) took 51.39 seconds
INFO 07-06 14:04:09 [core_cli

In [7]:

llm_results = process_questions_with_llm(filtered, llm, sampling_params, num_questions=50)

well_specified = [r for r in llm_results if r["is_well_specified"] == True]
vague_questions = [r for r in llm_results if r["is_well_specified"] == False]

print(f"\n" + "="*60)
print("ANALYSIS SUMMARY")
print("="*60)
print(f"Total questions analyzed: {len(llm_results)}")
print(f"Well-specified questions: {len(well_specified)}")
print(f"Vague questions: {len(vague_questions)}")

if well_specified:
    print(f"\nWELL-SPECIFIED QUESTIONS:")
    print("-" * 50)
    for i, result in enumerate(well_specified, 1):
        print(f"{i}. Original: {result['original_question']}")
        print()

if vague_questions:
    print(f"\nVAGUE QUESTIONS WITH CLARIFYING SUGGESTIONS:")
    print("-" * 50)
    for i, result in enumerate(vague_questions, 1):
        print(f"{i}. Original: {result['original_question']}")
        print(f"   Clarifying: {result['clarifying_question']}")
        print()


ANALYSIS SUMMARY
Total questions analyzed: 50
Well-specified questions: 42
Vague questions: 8

WELL-SPECIFIED QUESTIONS:
--------------------------------------------------
1. Original: Mera period parcel aata hai jab first din aata use dene ko black bleeding Hoti hai per uske paanchvein chhathe din se black bleeding Hoti hai bich mein meri red bleeding hoti

2. Original: Bachche ki heart beat or mother ki heart beat badhi hui he ,normal delivery me koi problem to nhi hoti he.or normal delivery ke kitne chance hote hei isme

3. Original: I am in 5th month of pregnancy and My mother inlaw does not allow me to go to doctor's appointments saying it is unnecessary and waste of money

4. Original: During 7 months pregnancy,if patient want more water drink.is it suger charector? Her leaps also dray due to water drink. Please say about it.

5. Original: Mera baby ka right  kidney scene me nei dekh roha hey aur pet me pani nei pouch roha hey sas lene toklip ho roha hey doctor bola

6. Original

In [10]:
questions_list = ['My baby weight is 452 gram in 22weeks..is this normal', 
             'Sometimes I don’t feel the baby’s move ..is it okay or something else?', 
             'I have seen prickly heat and itching on some part of my body... What should I do?',
             'Pragnancy me mango 🥭 kitna mntra me kha skti hu',
             'Bacche ko Tej Sardi ho to kaise usko maintain Karen',
             'Kya pregnancy m bleeding hoti h kya yeh normal h',
             'mujhe thyroid h to kya isse bhi breast milk pe asar hota h',
             '2.5 months hogae h pregnancy ko but vomet ya kch ni hora mjhe',
             'जयदा टाइम तक बैठने से चक्कर आना खड़ा होने से पेट दर्द होने लगता हैमेरे को कौन सी टेबलेट खानी चाहिए अभी सट्टा मा लग चुका है',
             'सर्दी -जुकाम  हो रहा  है।इसका बच्चे पर तो कोई प्रभाव  नहीं  होगा ।'] + filtered

In [ ]:
client = OpenAI()
# Create prompts for vague and specific versions
vague_prompt = """Make this health question more vague by removing specific details and making it harder to answer accurately without clarification. Keep the core concern and point of view. The vague version should be approximately the same length as the original question.

Original question: {question}

Vague version:"""

specific_prompt = """Make this health question more specific by adding reasonable details that would help a medical professional give a more accurate answer. Keep the core concern and point of view. The specific version should be approximately the same length as the original question.

Original question: {question}

Specific version:"""

# Initialize lists for vague and specific questions
vague_questions_list = []
specific_questions_list = []

# Process each question to make vague and specific versions
for question in tqdm(questions_list):
    # Get vague version from LLM
    vague_output = [
        client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": vague_prompt.format(question=question)}]
            ).choices[0].message.content
        ]
    vague_version = vague_output[0].strip()
    vague_questions_list.append(vague_version)
    
    # Get specific version from LLM
    specific_output = [
        client.chat.completions.create(
                model="gpt-4o-2024-11-20", 
                messages=[{"role": "user", "content": specific_prompt.format(question=question)}]
            ).choices[0].message.content
        ]
    specific_version = specific_output[0].strip()
    specific_questions_list.append(specific_version)

# Print all versions for comparison
print("\nOriginal vs Vague vs Specific Questions:")
print("-" * 50)
for orig, vague, specific in zip(questions_list, vague_questions_list, specific_questions_list):
    print(f"Original: {orig}")
    print(f"Vague: {vague}")
    print(f"Specific: {specific}\n")


  0%|          | 0/252 [00:00<?, ?it/s]

 27%|██▋       | 69/252 [02:54<06:35,  2.16s/it]

In [20]:
import csv
from datetime import datetime

def classify_question_clarity(client, question, model_name):
    """
    Use GPT model to classify if a question is well-specified or vague.
    """
    prompt = f"""You are a medical health chatbot answering user's questions about pregnancy. Analyze the following question and determine if it's well-specified or too vague to answer properly.

    Question: "{question}"

    Instructions:
    - If the question is well-specified (clear, specific, and answerable), respond with: "WELL_SPECIFIED"
    - If the question is too vague (lacks context, specificity, or is ambiguous), respond with: "VAGUE"

    Response:"""

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,  # Low temperature for consistent classification
            max_tokens=10
        )
        
        classification = response.choices[0].message.content.strip().upper()
        
        # Ensure we only return valid classifications
        if "WELL_SPECIFIED" in classification:
            return "WELL_SPECIFIED"
        elif "VAGUE" in classification:
            return "VAGUE"
        else:
            return "UNCLEAR_RESPONSE"
            
    except Exception as e:
        print(f"Error classifying question with {model_name}: {e}")
        return "ERROR"

def evaluate_models_on_questions():
    """
    Evaluate multiple GPT models on question classification task.
    """
    # Models to test
    models = ["gpt-4o", "gpt-4-turbo", "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1-nano"] 
    
    # Combine all questions with their expected classifications
    all_questions = []
    
    # Original questions (don't contribute to score)
    for q in questions_list:
        all_questions.append({
            'question': q,
            'question_type': 'original',
            'expected_classification': None  # No expected classification for scoring
        })
    
    # Vague questions (should be classified as VAGUE)
    for q in vague_questions_list:
        all_questions.append({
            'question': q,
            'question_type': 'vague',
            'expected_classification': 'VAGUE'
        })
    
    # Specific questions (should be classified as WELL_SPECIFIED)
    for q in specific_questions_list:
        all_questions.append({
            'question': q,
            'question_type': 'specific',
            'expected_classification': 'WELL_SPECIFIED'
        })
    
    results = []
    model_scores = {model: 0 for model in models}
    total_scoreable = len(vague_questions_list) + len(specific_questions_list)
    
    print(f"Evaluating {len(all_questions)} questions with {len(models)} models...")
    print(f"Total scoreable questions: {total_scoreable}")
    
    for i, question_data in tqdm(enumerate(all_questions)):
        question = question_data['question']
        question_type = question_data['question_type']
        expected = question_data['expected_classification']
        
        print(f"Processing question {i+1}/{len(all_questions)}...")
        
        row = {
            'question_index': i+1,
            'question': question,
            'question_type': question_type,
            'expected_classification': expected if expected else 'N/A'
        }
        
        # Get classification from each model
        for model in models:
            classification = classify_question_clarity(client, question, model)
            row[f'{model}_classification'] = classification
            
            # Calculate score for this model on this question
            score = 0
            if expected:  # Only score if we have an expected classification
                if classification == expected:
                    score = 1
                    model_scores[model] += 1
            
            row[f'{model}_score'] = score if expected else 'N/A'
        
        results.append(row)
    
    # Calculate final scores as percentages
    final_scores = {}
    for model in models:
        percentage = (model_scores[model] / total_scoreable) * 100
        final_scores[model] = {
            'correct': model_scores[model],
            'total': total_scoreable,
            'percentage': percentage
        }
    
    return results, final_scores

# Run the evaluation
print("Starting model evaluation...")
results, final_scores = evaluate_models_on_questions()


Starting model evaluation...
Evaluating 30 questions with 5 models...
Total scoreable questions: 20
Processing question 1/30...


[2025-07-06 15:19:10] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:11] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:11] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:12] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:12] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing question 2/30...


[2025-07-06 15:19:12] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:13] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:14] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:14] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:19:15] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing question 3/30...


...

[2025-07-06 15:20:16] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:16] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:17] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:18] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:18] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing question 29/30...


[2025-07-06 15:20:18] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:19] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:20] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:21] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:21] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing question 30/30...


[2025-07-06 15:20:22] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:22] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:23] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:23] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2025-07-06 15:20:23] INFO _client.py:1038: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [22]:
# Save results to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"model_evaluation_results_{timestamp}.csv"

# Create a list to store all rows
all_rows = []

# Add model results rows
for result in results:
    all_rows.append(result)

# Add blank row
all_rows.append({})

# Add summary scores
all_rows.append({'question': 'FINAL SCORES'})
for model, score_data in final_scores.items():
    all_rows.append({
        'question': model,
        'question_type': f"{score_data['correct']}/{score_data['total']} correct",
        'expected_classification': f"{score_data['percentage']:.1f}% accuracy"
    })

# Define CSV fieldnames
fieldnames = ['question_index', 'question', 'question_type', 'expected_classification']
models = ["gpt-4o", "gpt-4-turbo", "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1-nano"] 
for model in models:
    fieldnames.extend([f'{model}_classification', f'{model}_score'])

# Write all results to CSV
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"All results and summary scores saved to: {csv_filename}")


All results and summary scores saved to: model_evaluation_results_20250706_152235.csv
